In [18]:
import os
import re
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from docx import Document
from docx.enum.style import WD_STYLE_TYPE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ollama
import time

In [1]:
!pip install PySide6

   ---------------------------------------- 0.0/578.0 kB ? eta -:--:--
   ------------------ --------------------- 262.1/578.0 kB ? eta -:--:--
   ---------------------------------------- 578.0/578.0 kB 3.4 MB/s  0:00:00
   ---------------------------------------- 0.0/168.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/168.7 MB 4.8 MB/s eta 0:00:36
    --------------------------------------- 2.4/168.7 MB 6.4 MB/s eta 0:00:27
    --------------------------------------- 3.7/168.7 MB 6.2 MB/s eta 0:00:27
   - -------------------------------------- 5.5/168.7 MB 6.8 MB/s eta 0:00:24
   - -------------------------------------- 6.6/168.7 MB 6.5 MB/s eta 0:00:25
   - -------------------------------------- 7.9/168.7 MB 6.4 MB/s eta 0:00:26
   - -------------------------------------- 8.4/168.7 MB 5.9 MB/s eta 0:00:28
   -- ------------------------------------- 9.4/168.7 MB 5.8 MB/s eta 0:00:28
   -- ------------------------------------- 10.0/168.7 MB 5.5 MB/s eta 0:00:29
   --


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Architecture

# Folder Structure

# Mock-Up Reference Document

# Template Structure

## Step 1: Load in documents

In [2]:
#########################################################
# STEP 1 — LOAD DOCUMENTS
#########################################################

def load_word_documents(folder_path="documents"):

    documents = []

    for filename in os.listdir(folder_path):

        if filename.endswith(".docx"):

            path = os.path.join(folder_path, filename)

            doc = Document(path)

            documents.append({
                "filename": filename,
                "doc": doc
            })

    return documents


## Step 2: Divide text into distinct chunks or sections

In [3]:
#########################################################
# STEP 2 — SMART CHUNKING
#########################################################

def extract_structured_sections(documents):

    structured_sections = []

    for document in documents:

        filename = document["filename"]
        doc = document["doc"]

        current_h1 = None
        current_h2 = None
        current_h3 = None

        current_content = []

        def save_section():

            if current_content:

                structured_sections.append({
                    "filename": filename,
                    "heading_1": current_h1,
                    "heading_2": current_h2,
                    "heading_3": current_h3,
                    "content": "".join(current_content)
                })

        for para in doc.paragraphs:

            text = para.text.strip()

            if not text:
                continue

            style_name = para.style.name.lower()

            # HEADING 1 / RUBRIK 1
            if "heading 1" in style_name or "rubrik 1" in style_name:

                save_section()

                current_h1 = text
                current_h2 = None
                current_h3 = None
                current_content = []

            # HEADING 2 / RUBRIK 2
            elif "heading 2" in style_name or "rubrik 2" in style_name:

                save_section()

                current_h2 = text
                current_h3 = None
                current_content = []

            # HEADING 3 / RUBRIK 3
            elif "heading 3" in style_name or "rubrik 3" in style_name:

                save_section()

                current_h3 = text
                current_content = []

            else:
                current_content.append(text)

        save_section()

    return structured_sections
def build_procedural_blocks(sections):

    procedural_blocks = []

    for i, section in enumerate(sections):

        block_text = f"""
DOCUMENT: {section['filename']}

MAIN SECTION:
{section['heading_1']}

SUBSECTION:
{section['heading_2']}

DETAIL SECTION:
{section['heading_3']}

CONTENT:
{section['content']}
"""

        procedural_blocks.append({
            "id": i,
            "text": block_text,
            "metadata": section
        })

    return procedural_blocks

## Step 3: Build vector index

In [4]:
#########################################################
# STEP 3 — BUILD VECTOR INDEX
#########################################################

def build_hybrid_index(blocks):

    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

    texts = [b["text"] for b in blocks]

    embeddings = embedding_model.encode(texts, show_progress_bar=True)

    dim = embeddings.shape[1]

    faiss_index = faiss.IndexFlatL2(dim)
    faiss_index.add(np.array(embeddings))

    # BM25-lite using TF-IDF
    tfidf_vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = tfidf_vectorizer.fit_transform(texts)

    return {
        "faiss_index": faiss_index,
        "embedding_model": embedding_model,
        "texts": texts,
        "blocks": blocks,
        "tfidf_vectorizer": tfidf_vectorizer,
        "tfidf_matrix": tfidf_matrix
    }

## Step 4: Retrieve the required context to generate an accurate text

In [5]:
#########################################################
# STEP 4 — RETRIEVE RELEVANT CONTEXT
#########################################################

def expand_query(user_query):

    expansions = {
        "startup": [
            "startup sequence",
            "activation procedure",
            "system initialization"
        ],

        "shutdown": [
            "cooling sequence",
            "pressure reduction",
            "shutdown verification"
        ],

        "safety": [
            "emergency shutdown",
            "protective equipment",
            "hazard prevention"
        ]
    }

    expanded_queries = [user_query]

    lower_query = user_query.lower()

    for keyword, related in expansions.items():

        if keyword in lower_query:
            expanded_queries.extend(related)

    return expanded_queries

## Step 5: Hybrid retrieval

In [6]:
############################################################
# STEP 6 — HYBRID RETRIEVAL
############################################################


def hybrid_retrieve(query, retrieval_system, k=10):

    faiss_index = retrieval_system["faiss_index"]
    embedding_model = retrieval_system["embedding_model"]
    texts = retrieval_system["texts"]
    blocks = retrieval_system["blocks"]

    tfidf_vectorizer = retrieval_system["tfidf_vectorizer"]
    tfidf_matrix = retrieval_system["tfidf_matrix"]

    expanded_queries = expand_query(query)

    all_scores = {}

    for q in expanded_queries:

        # VECTOR SEARCH
        query_embedding = embedding_model.encode([q])

        distances, indices = faiss_index.search(query_embedding, k)

        for rank, idx in enumerate(indices[0]):

            score = 1 / (1 + distances[0][rank])

            all_scores[idx] = all_scores.get(idx, 0) + score

        # KEYWORD SEARCH
        query_tfidf = tfidf_vectorizer.transform([q])

        cosine_scores = cosine_similarity(
            query_tfidf,
            tfidf_matrix
        )[0]

        for idx, score in enumerate(cosine_scores):
            all_scores[idx] = all_scores.get(idx, 0) + score

    ranked = sorted(
        all_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    retrieved = []

    for idx, score in ranked[:k]:

        block = blocks[idx]

        retrieved.append({
            "score": score,
            "text": texts[idx],
            "metadata": block["metadata"]
        })

    return retrieved

## Step 6: context assembly

In [7]:
############################################################
# STEP 7 — HIERARCHICAL CONTEXT ASSEMBLY
############################################################


def build_structured_context(retrieved_blocks):

    context = ""

    for i, block in enumerate(retrieved_blocks):

        metadata = block["metadata"]

        context += f"""
====================================================
REFERENCE BLOCK {i+1}
====================================================

SOURCE FILE:
{metadata['filename']}

MAIN HEADING:
{metadata['heading_1']}

SUB HEADING:
{metadata['heading_2']}

DETAIL HEADING:
{metadata['heading_3']}

CONTENT:
{metadata['content']}

"""

    return context

## Step 6: Decide on a template

In [8]:
############################################################
# STEP 8 — SECTION-AWARE RETRIEVAL
############################################################


def retrieve_by_procedure_section(user_request, retrieval_system):

    retrieval_map = {
        "Preconditions": [
            "inspection requirements",
            "startup verification",
            user_request
        ],

        "Safety Precautions": [
            "safety requirements",
            "emergency shutdown",
            "protective equipment"
        ],

        "Procedure Steps": [
            "startup sequence",
            "activation procedure",
            user_request
        ],

        "Validation Checks": [
            "verification",
            "validation",
            "stability checks"
        ],

        "Escalation Conditions": [
            "failure conditions",
            "escalation",
            "critical damage"
        ]
    }

    section_context = {}

    for section, queries in retrieval_map.items():

        combined_results = []

        for query in queries:

            results = hybrid_retrieve(
                query,
                retrieval_system,
                k=5
            )

            combined_results.extend(results)

        # RERANK + REMOVE DUPLICATES
        unique_results = {}

        for result in combined_results:

            key = result["text"]

            if key not in unique_results:
                unique_results[key] = result

        reranked = sorted(
            unique_results.values(),
            key=lambda x: x["score"],
            reverse=True
        )

        section_context[section] = reranked[:5]

    return section_context

## Step 7: Create the prompt and personality of LLM

In [9]:
############################################################
# STEP 9 — PROCEDURE GENERATION
############################################################


def generate_general_procedure(user_request, section_context):

    assembled_context = ""

    for section_name, references in section_context.items():

        assembled_context += f"""
##################################################
SECTION CONTEXT: {section_name}
##################################################
"""

        assembled_context += build_structured_context(references)

    prompt = f"""
You are a senior enterprise technical procedure writer.

Your task is to generate a professional General Procedure.

CRITICAL RULES:
- ONLY use information from retrieved references
- NEVER hallucinate operational details
- If information is missing, explicitly state so
- Maintain procedural flow and operational sequencing
- Use standardized technical English
- Generate logically ordered operational steps

The procedure should:
1. Begin with Preconditions
2. Continue through Safety Precautions
3. Continue through Procedure Steps
4. Include Validation Checks
5. Include Escalation Conditions

USER REQUEST:
{user_request}

RETRIEVED CONTEXT:
{assembled_context}

OUTPUT TEMPLATE:

1. Objective
2. Scope
3. Preconditions
4. Required Equipment
5. Safety Precautions
6. Procedure Steps
7. Validation Checks
8. Escalation Conditions

VERY IMPORTANT:
- Cite heading references when possible
- Use operational language
- Generate complete procedural flow
- Keep procedural numbering consistent
"""

    response = ollama.chat(
        model="llama3",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

## Step 8: Give personality to LLM who is goiung to verify the generated text

In [24]:
############################################################
# STEP 10 — VALIDATION AGENT
############################################################


def validate_procedure(procedure_text, section_context):

    context = ""

    for refs in section_context.values():
        context += build_structured_context(refs)

    prompt = f"""
You are a senior enterprise reviewer responsible for validating
General Procedures used in industrial operations.

IMPORTANT CONTEXT:

A General Procedure is:
- a high-level standardized operational procedure
- intended to guide consistent execution of routine activities
- not necessarily a detailed engineering manual
- not expected to contain every possible operational edge case
- focused on operational clarity, sequencing, safety, and consistency

Your role is to validate whether the generated General Procedure:

1. Remains grounded in the retrieved reference material
2. Avoids unsupported operational claims
3. Maintains logical operational sequencing
4. Includes appropriate safety considerations
5. Uses clear and professional technical language
6. Is sufficiently complete for a General Procedure document

VERY IMPORTANT VALIDATION RULES:

- DO NOT fail the procedure simply because certain details are absent
  from the source documents.

- ONLY flag missing information if:
  a) the information is clearly required for procedural safety or logic
  AND
  b) the information appears to be implied or partially referenced
      in the provided context.

- A General Procedure is allowed to:
  - remain high-level
  - omit engineering-level implementation details
  - omit undefined system internals
  - omit detailed troubleshooting logic

- DO NOT invent missing requirements.

- DO NOT criticize the procedure for lacking information that does
  not exist in the reference context.

- DO NOT generate hypothetical operational concerns.

- Be conservative when flagging issues.

REFERENCE CONTEXT:
{context}

GENERATED GENERAL PROCEDURE:
{procedure_text}

VALIDATION TASKS:

1. Unsupported Statements
Identify statements that are NOT supported by the retrieved references.

2. Critical Missing Information
Identify ONLY operationally important missing information that:
- appears implied by the references
- affects safety, sequencing, or procedural correctness

3. Ambiguous or Unclear Wording
Identify wording that could realistically confuse operators.

4. Procedural Consistency
Check whether the procedure follows logical operational flow.

5. Overall Assessment
Provide a concise enterprise-style assessment.

6. Validation Result
Return ONE of:
- PASS
- PASS WITH MINOR REVISIONS
- FAIL

IMPORTANT:
- Be strict but realistic.
- Evaluate this as a General Procedure,
  NOT as a detailed engineering specification.
"""

    response = ollama.chat(
        model="llama3",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

## Step 9: Give personality to LLM that is going to rewrite it into Technical English

In [25]:
############################################################
# STEP 11 — EXPORT WORD REPORT
############################################################


def save_word_report(
    procedure_text,
    validation_text,
    output_path="output/generated_procedure.docx",
    output_path2="output/validation.docx"
):
    #Validation document
    doc = Document()

    doc.add_heading("Generated General Procedure", level=1)

    doc.add_heading("Procedure", level=2)
    doc.add_paragraph(procedure_text)
    doc.save(output_path)
    
    #Validation document
    doc2 = Document()
    doc2.add_heading("Validation Results", level=2)
    doc2.add_paragraph(validation_text)

    doc2.save(output_path2)

    return output_path, output_path2

## Step 10: Save text into word format.

In [26]:
############################################################
# STEP 12 — FULL WORKFLOW
############################################################


def run_workflow(user_request):
    iteration_times = []
    start_time = time.time()
    print("STEP 1 — Loading Word documents")

    documents = load_word_documents()

    print("STEP 2 — Extracting heading structure")

    sections = extract_structured_sections(documents)

    print("STEP 3 — Building procedural blocks")

    procedural_blocks = build_procedural_blocks(sections)

    print("STEP 4 — Building hybrid retrieval system")
   
    retrieval_system = build_hybrid_index(procedural_blocks)

    print("STEP 5 — Intelligent section-aware retrieval")

    section_context = retrieve_by_procedure_section(
        user_request,
        retrieval_system
    )

    print("STEP 6 — Generating General Procedure")

    procedure = generate_general_procedure(
        user_request,
        section_context
    )

    print("STEP 7 — Validating procedure")

    validation = validate_procedure(
        procedure,
        section_context
    )

    print("STEP 8 — Saving Word report")

    output_path, output_path2 = save_word_report(
        procedure,
        validation
    )

    print("DONE")
    total_time = time.time() - start_time
    print(f"\n⏳ General Procedure Generation completed in {round(total_time, 2)} seconds.")
    return {
        "procedure": procedure,
        "validation": validation,
        "output_path": output_path,
        "output_path_validation": output_path2
    }




## Step 11: Run the workflow

In [28]:
############################################################
# EXAMPLE EXECUTION
############################################################

if __name__ == "__main__":

    result = run_workflow(
        "Generate a startup procedure for the PX-400 pressure system"
    )

    print(result["procedure"])

STEP 1 — Loading Word documents
STEP 2 — Extracting heading structure
STEP 3 — Building procedural blocks
STEP 4 — Building hybrid retrieval system


Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 11.57it/s]


STEP 5 — Intelligent section-aware retrieval
STEP 6 — Generating General Procedure
STEP 7 — Validating procedure
STEP 8 — Saving Word report
DONE

⏳ General Procedure Generation completed in 20.12 seconds.
**General Procedure: Startup of PX-400 Pressure System**

**Objective:** To provide a standardized procedure for the startup of the PX-400 pressure system.

**Scope:** This document describes the startup procedures for the PX-400 pressure system.

**Preconditions:**

1. The primary control panel must be activated, and indicator lights must be green.
2. Operators must inspect pressure gauges, valve seals, and cooling systems before startup.

**Required Equipment:** None specified

**Safety Precautions:**

1. Emergency shutdown access points must remain unobstructed (Reference Block 1).
2. Pressure must be reduced gradually during shutdown (Reference Block 2).
3. Cooling sequence must complete before maintenance begins (Reference Block 4).
4. Operators must wear protective gloves, safe

In [ ]:
result

# Version 2

In [37]:
# Enterprise Procedural RAG Pipeline — Requirement-Driven Architecture

############################################################
# IMPORTS
############################################################

import os
import re
import json
import faiss
import numpy as np
import pandas as pd
from docx import Document
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ollama
import time

############################################################
# EMBEDDING MODEL
############################################################

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

############################################################
# STEP 1 — WORD DOCUMENT PARSING
############################################################


def parse_word_document(filepath):

    doc = Document(filepath)

    sections = []

    current_h1 = None
    current_h2 = None
    current_content = []

    def flush_section():
        nonlocal current_content

        if current_content:
            sections.append({
                "heading_1": current_h1,
                "heading_2": current_h2,
                "content": "\n".join(current_content).strip(),
                "document": os.path.basename(filepath)
            })
            current_content = []

    for para in doc.paragraphs:

        text = para.text.strip()

        if not text:
            continue

        style_name = para.style.name.lower()
        style_name = re.sub(r"\s+", " ", style_name)

        # Heading 1 / Rubrik 1
        if "heading 1" in style_name or "rubrik 1" in style_name:
            flush_section()
            current_h1 = text
            current_h2 = None

        # Heading 2 / Rubrik 2
        elif "heading 2" in style_name or "rubrik 2" in style_name:
            flush_section()
            current_h2 = text

        else:
            current_content.append(text)

    flush_section()

    return sections


############################################################
# STEP 2 — LOAD ALL DOCUMENTS
############################################################


def load_all_documents(folder_path):

    all_sections = []

    for file in os.listdir(folder_path):

        if file.endswith(".docx"):

            path = os.path.join(folder_path, file)

            sections = parse_word_document(path)

            all_sections.extend(sections)

    return all_sections


############################################################
# STEP 3 — REQUIREMENT EXTRACTION
############################################################


def extract_requirements(section, model_name="llama3"):

    prompt = f"""
You are an industrial operations analyst.

Extract ALL operational requirements from the section below.

SECTION:
{section['content']}

For EACH requirement return JSON with:

- requirement_type
  (safety, validation, operational, escalation, equipment)

- operational_phase
  (startup, shutdown, maintenance, emergency, inspection, general)

- requirement_statement

- criticality
  (high, medium, low)

IMPORTANT:
- Extract explicit requirements only
- Do NOT summarize
- Do NOT invent information
- Return ONLY valid JSON array
- No explanations
- No markdown
"""

    response = ollama.chat(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    content = response["message"]["content"]

    ########################################################
    # CLEAN RESPONSE
    ########################################################

    # Remove markdown fences
    content = re.sub(r"```json", "", content)
    content = re.sub(r"```", "", content)

    ########################################################
    # EXTRACT JSON ARRAY
    ########################################################

    match = re.search(r"\[.*\]", content, re.DOTALL)

    if not match:
        print("FAILED JSON EXTRACTION")
        print(content)
        return []

    json_text = match.group(0)

    ########################################################
    # PARSE JSON
    ########################################################

    try:

        requirements = json.loads(json_text)

        ####################################################
        # ADD METADATA
        ####################################################

        for r in requirements:

            r["source_heading_1"] = section["heading_1"]
            r["source_heading_2"] = section["heading_2"]
            r["source_document"] = section["document"]

        return requirements

    except Exception as e:

        print("JSON PARSE FAILED")
        print(e)
        print(json_text)

        return []


############################################################
# STEP 4 — BUILD REQUIREMENT DATABASE
############################################################


def build_requirement_database(sections):

    all_requirements = []

    for section in sections:

        if not section["content"].strip():
            continue

        reqs = extract_requirements(section)

        all_requirements.extend(reqs)

    return all_requirements


############################################################
# STEP 5 — DEDUPLICATION
############################################################


def deduplicate_requirements(requirements, threshold=0.90):

    if not requirements:
        return []

    texts = [r["requirement_statement"] for r in requirements]

    embeddings = embedding_model.encode(texts)

    unique_requirements = []
    used = set()

    for i in range(len(requirements)):

        if i in used:
            continue

        unique_requirements.append(requirements[i])

        for j in range(i + 1, len(requirements)):

            sim = cosine_similarity(
                [embeddings[i]],
                [embeddings[j]]
            )[0][0]

            if sim >= threshold:
                used.add(j)

    return unique_requirements


############################################################
# STEP 6 — BUILD REQUIREMENT RETRIEVAL INDEX
############################################################


def build_requirement_index(requirements):

    texts = [r["requirement_statement"] for r in requirements]

    if len(texts) == 0:
        raise ValueError("No requirements found.")

    embeddings = embedding_model.encode(texts)
    embeddings = np.array(embeddings)

    if len(embeddings.shape) == 1:
        embeddings = embeddings.reshape(1, -1)

    dim = embeddings.shape[1]

    faiss_index = faiss.IndexFlatL2(dim)
    faiss_index.add(embeddings)

    tfidf = TfidfVectorizer()
    tfidf_matrix = tfidf.fit_transform(texts)

    return {
        "requirements": requirements,
        "texts": texts,
        "faiss_index": faiss_index,
        "embeddings": embeddings,
        "tfidf": tfidf,
        "tfidf_matrix": tfidf_matrix
    }


############################################################
# STEP 7 — PROCEDURE TYPE DETECTION
############################################################


def detect_procedure_type(user_request):

    text = user_request.lower()

    if "startup" in text:
        return "startup"

    if "shutdown" in text:
        return "shutdown"

    if "maintenance" in text:
        return "maintenance"

    if "inspection" in text:
        return "inspection"

    return "general"


############################################################
# STEP 8 — REQUIREMENT RETRIEVAL
############################################################


def retrieve_requirements(
    query,
    retrieval_system,
    procedure_type,
    requirement_type=None,
    k=10
):

    requirements = retrieval_system["requirements"]

    filtered_indices = []

    for idx, req in enumerate(requirements):

        phase = req.get("operational_phase", "general")

        phase_match = (
            phase == procedure_type
            or phase == "general"
        )

        type_match = (
            requirement_type is None
            or req["requirement_type"] == requirement_type
        )

        if phase_match and type_match:
            filtered_indices.append(idx)

    if not filtered_indices:
        return []

    filtered_texts = [
        requirements[i]["requirement_statement"]
        for i in filtered_indices
    ]

    filtered_embeddings = embedding_model.encode(filtered_texts)

    query_embedding = embedding_model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        filtered_embeddings
    )[0]

    top_indices = np.argsort(similarities)[::-1][:k]

    results = []

    for idx in top_indices:

        req = requirements[filtered_indices[idx]]

        results.append(req)

    return results


############################################################
# STEP 9 — FORMAT REQUIREMENTS
############################################################


def format_requirements(requirements):

    text = ""

    for i, req in enumerate(requirements):

        text += f"""
Requirement {i+1}
Type: {req['requirement_type']}
Phase: {req['operational_phase']}
Criticality: {req['criticality']}
Statement: {req['requirement_statement']}
Source: {req['source_document']}
Heading 1: {req['source_heading_1']}
Heading 2: {req['source_heading_2']}

"""

    return text


############################################################
# STEP 10 — SECTION GENERATION
############################################################


def generate_section(
    section_name,
    requirements,
    user_request,
    model_name="llama3"
):

    requirement_text = format_requirements(requirements)

    prompt = f"""
You are writing a professional enterprise General Procedure.

TASK:
Generate ONLY the section:
{section_name}

USER REQUEST:
{user_request}

REQUIREMENTS:
{requirement_text}

RULES:
- Use ONLY the provided requirements
- Do NOT invent information
- Use concise technical language
- Maintain professional formatting
- If no relevant requirements exist, state:
  'No specific requirements identified.'
- Do NOT generate other sections
"""

    response = ollama.chat(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]


############################################################
# STEP 11 — FULL PROCEDURE GENERATION
############################################################


def generate_general_procedure(
    user_request,
    retrieval_system
):

    procedure_type = detect_procedure_type(user_request)

    section_map = {
        "Objective": None,
        "Scope": None,
        "Preconditions": "operational",
        "Required Equipment": "equipment",
        "Safety Precautions": "safety",
        "Procedure Steps": "operational",
        "Validation Checks": "validation",
        "Escalation Conditions": "escalation"
    }

    generated_sections = {}

    for section_name, req_type in section_map.items():

        reqs = retrieve_requirements(
            query=user_request,
            retrieval_system=retrieval_system,
            procedure_type=procedure_type,
            requirement_type=req_type,
            k=8
        )

        generated_sections[section_name] = generate_section(
            section_name,
            reqs,
            user_request
        )

    final_document = f"""
# General Procedure

## Objective
{generated_sections['Objective']}

## Scope
{generated_sections['Scope']}

## Preconditions
{generated_sections['Preconditions']}

## Required Equipment
{generated_sections['Required Equipment']}

## Safety Precautions
{generated_sections['Safety Precautions']}

## Procedure Steps
{generated_sections['Procedure Steps']}

## Validation Checks
{generated_sections['Validation Checks']}

## Escalation Conditions
{generated_sections['Escalation Conditions']}
DO NOT:
- explain benefits
- justify procedure
- describe operational philosophy
- add industry assumptions
- add generic engineering language

ONLY write operationally actionable content directly supported by requirements.
"""

    return final_document


############################################################
# STEP 12 — REQUIREMENT COVERAGE VALIDATION
############################################################


def validate_requirement_coverage(
    procedure_text,
    requirements,
    model_name="llama3"
):

    requirement_text = format_requirements(requirements)

    prompt = f"""
You are validating requirement coverage.

REQUIREMENTS:
{requirement_text}

PROCEDURE:
{procedure_text}

TASK:
For EACH requirement determine:
- Included
- Partially Included
- Missing

Return JSON list.
"""

    response = ollama.chat(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]


############################################################
# STEP 13 — ENTERPRISE VALIDATION
############################################################


def validate_procedure(
    procedure_text,
    requirements,
    model_name="llama3"
):

    context = format_requirements(requirements)

    prompt = f"""
You are a senior enterprise reviewer responsible for validating
General Procedures used in industrial operations.

IMPORTANT CONTEXT:

A General Procedure is:
- a high-level standardized operational procedure
- intended to guide consistent execution of routine activities
- not necessarily a detailed engineering manual
- focused on operational clarity, sequencing, safety, and consistency

REFERENCE CONTEXT:
{context}

GENERATED GENERAL PROCEDURE:
{procedure_text}

VALIDATION TASKS:

1. Unsupported Statements
2. Critical Missing Information
3. Ambiguous Wording
4. Procedural Consistency
5. Overall Assessment
6. Validation Result

IMPORTANT:
- Do NOT invent missing requirements
- Evaluate this as a General Procedure
- Be strict but realistic
"""

    response = ollama.chat(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]


############################################################
# STEP 14 — SAVE WORD DOCUMENT
############################################################



def save_procedure_docx(
    procedure_text,
    validation_text,
    output_path="output/generated_procedure.docx",
    output_path2="output/validation.docx"
):
    #Validation document
    doc = Document()

    doc.add_heading("Generated General Procedure", level=1)

    doc.add_heading("Procedure", level=2)
    doc.add_paragraph(procedure_text)
    doc.save(output_path)
    
    #Validation document
    doc2 = Document()
    doc2.add_heading("Validation Results", level=2)
    doc2.add_paragraph(validation_text)

    doc2.save(output_path2)

    return output_path, output_path2

############################################################
# STEP 15 — FULL WORKFLOW
############################################################


def run_workflow(
    user_request,
    document_folder
):
    iteration_times = []
    start_time = time.time()
    print("STEP 1 — Loading documents")

    sections = load_all_documents(document_folder)

    print(f"Loaded sections: {len(sections)}")

    print("STEP 2 — Extracting requirements")

    requirements = build_requirement_database(sections)

    print(f"Extracted requirements: {len(requirements)}")

    print("STEP 3 — Deduplicating requirements")

    requirements = deduplicate_requirements(requirements)

    print(f"Unique requirements: {len(requirements)}")

    print("STEP 4 — Building retrieval system")

    retrieval_system = build_requirement_index(requirements)

    print("STEP 5 — Generating procedure")

    procedure = generate_general_procedure(
        user_request,
        retrieval_system
    )

    print("STEP 6 — Requirement coverage validation")

    coverage = validate_requirement_coverage(
        procedure,
        requirements
    )

    print("STEP 7 — Enterprise validation")

    validation = validate_procedure(
        procedure,
        requirements
    )

    print("STEP 8 — Saving document")

    output_path, output_path2 = save_procedure_docx(
        procedure,
        validation
    )
    print("DONE")
    total_time = time.time() - start_time
    print(f"\n⏳ General Procedure Generation completed in {round(total_time, 2)} seconds.")

    return {
        "procedure": procedure,
        "coverage": coverage,
        "validation": validation,
        "output_path": output_path,
        "output_path_validation": output_path2,
        "requirements": requirements
    }





F:\Pytorch\venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [38]:
############################################################
# EXAMPLE EXECUTION
############################################################


if __name__ == "__main__":

    result = run_workflow(
        user_request="Generate startup procedure for PX-400 pressure system",
        document_folder="documents"
    )

    print(result["procedure"])
    print(result["validation"])


STEP 1 — Loading documents
Loaded sections: 22
STEP 2 — Extracting requirements
Extracted requirements: 37
STEP 3 — Deduplicating requirements
Unique requirements: 37
STEP 4 — Building retrieval system
STEP 5 — Generating procedure
STEP 6 — Requirement coverage validation
STEP 7 — Enterprise validation
STEP 8 — Saving document
DONE

⏳ General Procedure Generation completed in 113.67 seconds.

# General Procedure

## Objective
**Objective**

The objective of this startup procedure for the PX-400 pressure system is to ensure a safe and controlled startup process that meets all operational and safety requirements. This procedure outlines the necessary steps to initiate operation of the PX-400 pressure system, ensuring compliance with relevant requirements and minimizing risk.

Note: The following requirements have been identified as critical to the successful startup of the PX-400 pressure system:

* Requirement 1: Minimum 10-minute delay between start-up attempts (medium criticality)
* R